In [1]:
import numpy as np
import pandas as pd
from sqlalchemy import create_engine

In [2]:
data_01 = pd.read_csv('../data/raw/olist_customers_dataset.csv')
data_02 = pd.read_csv('../data/raw/olist_geolocation_dataset.csv')
data_03 = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
data_04 = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')
data_05 = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')
data_06 = pd.read_csv('../data/raw/olist_orders_dataset.csv')
data_07 = pd.read_csv('../data/raw/olist_products_dataset.csv')
data_08 = pd.read_csv('../data/raw/olist_sellers_dataset.csv')
data_09 = pd.read_csv('../data/raw/product_category_name_translation.csv')

In [3]:
for i in range(1,10) : 
    print(f'-------------data_0{i}----------------------')
    var_name = f"data_{i:02d}"
    getattr(globals()[var_name], "info")()

-------------data_01----------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB
-------------data_02----------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng      

In [4]:
data_06['order_purchase_timestamp'] = pd.to_datetime(data_06['order_purchase_timestamp'])
data_06['order_approved_at'] = pd.to_datetime(data_06['order_approved_at'])
data_06['order_delivered_carrier_date'] = pd.to_datetime(data_06['order_delivered_carrier_date'])
data_06['order_delivered_customer_date'] = pd.to_datetime(data_06['order_delivered_customer_date'])
data_06['order_estimated_delivery_date'] = pd.to_datetime(data_06['order_estimated_delivery_date'])

data_03['shipping_limit_date'] = pd.to_datetime(data_03['shipping_limit_date'])

In [5]:
orders_clean = data_06[data_06['order_status'] == 'delivered'].copy()

In [6]:
orders_clean = orders_clean[['order_id','customer_id','order_purchase_timestamp','order_status']]
customers_clean = data_01[['customer_id','customer_state','customer_city']]
items_clean = data_03[['order_id','product_id','seller_id','order_item_id','price','freight_value']]
products_clean = data_07[['product_id','product_category_name']]
translation_clean = data_09
sellers_clean = data_08[['seller_id','seller_state']]


#### Making Master Table

In [7]:
master_df = items_clean.merge(
    orders_clean,
    on='order_id',
    how='inner'
)

master_df = master_df.merge(
    products_clean,
    on='product_id',
    how='left'
)

master_df = master_df.merge(
    translation_clean,
    on='product_category_name',
    how='left'
)

master_df = master_df.merge(
    customers_clean,
    on='customer_id',
    how='left'
)

master_df = master_df.merge(
    sellers_clean,
    on='seller_id',
    how='left'
)

master_df['date'] = master_df['order_purchase_timestamp'].dt.date
master_df['date'] = pd.to_datetime(master_df['date'])



In [8]:
print(master_df.shape)
master_df.head()

(110197, 15)


,order_id,product_id,seller_id,order_item_id,price,freight_value,customer_id,order_purchase_timestamp,order_status,product_category_name,product_category_name_english,customer_state,customer_city,seller_state,date
0,00010242fe8c5a6d1ba2dd792cb16214,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,1,58.90,13.29,3ce436f183e68e07877b285a838db11a,2017-09-13 08:59:02,delivered,cool_stuff,cool_stuff,RJ,campos dos goytacazes,SP,2017-09-13
1,00018f77f2f0320c557190d7a144bdd3,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,1,239.90,19.93,f6dd3ec061db4e3987629fe6b26e5cce,2017-04-26 10:53:06,delivered,pet_shop,pet_shop,SP,santa fe do sul,SP,2017-04-26
2,000229ec398224ef6ca0657da4fc703e,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,1,199.00,17.87,6489ae5e4333f3693df5ad4372dab6d3,2018-01-14 14:33:31,delivered,moveis_decoracao,furniture_decor,MG,para de minas,MG,2018-01-14
3,00024acbcdf0a6daa1e931b038114c75,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,1,12.99,12.79,d4eb9395c8c0431ee92fce09860c5a06,2018-08-08 10:00:35,delivered,perfumaria,perfumery,SP,atibaia,SP,2018-08-08
4,00042b26cf59d7ce69dfabb4e55b4fd9,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,1,199.90,18.14,58dbd0b2d70206bf40e62cd34e84d795,2017-02-04 13:57:51,delivered,ferramentas_jardim,garden_tools,SP,varzea paulista,PR,2017-02-04


Retaining ID Columns: Integrity vs. Modeling
In the data integration stage, identifiers (e.g., order_id, customer_id) are kept in the Master Table to maintain relational structure and traceability, even though they are excluded from the final Modeling Dataset.

1. The Role of IDs in the Master Table
Identifiers act as primary and foreign keys, ensuring data integrity during the merging of relational tables (orders, products, sellers). They serve three critical functions:

Traceability: Allows for debugging and row-level validation back to source tables.

Data Quality: Enables precise duplicate detection and anomaly inspection.

Flexibility: Preserves the ability to pivot to new analyses, such as seller performance or customer lifetime value.

2. Why IDs are Excluded from Modeling
Identifiers are arbitrary labels and lack predictive power. Including them as features introduces noise and risks overfitting, as the model may incorrectly treat these unique strings or integers as meaningful numerical patterns.

3. Transition to Forecasting
For time-series forecasting, the data is aggregated by dimensions like date, category, and state. During this process, raw IDs are transformed into meaningful analytical variables:

order_item_id becomes Total Sales (count).

price becomes Total Revenue (sum).

In [9]:
master_df.to_csv('../data/interim/master_table.csv',index=False)

In [10]:
master_df.head()

,order_id,product_id,seller_id,order_item_id,price,freight_value,customer_id,order_purchase_timestamp,order_status,product_category_name,product_category_name_english,customer_state,customer_city,seller_state,date
0,00010242fe8c5a6d1ba2dd792cb16214,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,1,58.90,13.29,3ce436f183e68e07877b285a838db11a,2017-09-13 08:59:02,delivered,cool_stuff,cool_stuff,RJ,campos dos goytacazes,SP,2017-09-13
1,00018f77f2f0320c557190d7a144bdd3,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,1,239.90,19.93,f6dd3ec061db4e3987629fe6b26e5cce,2017-04-26 10:53:06,delivered,pet_shop,pet_shop,SP,santa fe do sul,SP,2017-04-26
2,000229ec398224ef6ca0657da4fc703e,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,1,199.00,17.87,6489ae5e4333f3693df5ad4372dab6d3,2018-01-14 14:33:31,delivered,moveis_decoracao,furniture_decor,MG,para de minas,MG,2018-01-14
3,00024acbcdf0a6daa1e931b038114c75,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,1,12.99,12.79,d4eb9395c8c0431ee92fce09860c5a06,2018-08-08 10:00:35,delivered,perfumaria,perfumery,SP,atibaia,SP,2018-08-08
4,00042b26cf59d7ce69dfabb4e55b4fd9,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,1,199.90,18.14,58dbd0b2d70206bf40e62cd34e84d795,2017-02-04 13:57:51,delivered,ferramentas_jardim,garden_tools,SP,varzea paulista,PR,2017-02-04
